# Entraînement sur Google Colab (GPU)

Ce notebook prépare Colab puis lance `train.py`. **Avant de commencer** :
`Exécution > Modifier le type d'exécution > GPU T4`.

1. Monter Google Drive (le zip des données et les résultats y sont).
2. Décompresser le zip sur le disque local de Colab (lire depuis Drive serait très lent).
3. Cloner le dépôt et installer `mne`.
4. Vérifier le GPU, puis lancer l'essai rapide, puis (après validation) l'entraînement complet.

Les résultats et les poids sont écrits sur Drive **après chaque pli**. Après une déconnexion,
il suffit de tout relancer : les plis déjà finis sont relus, pas recalculés.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paramètres (à adapter)

In [ ]:
ZIP_DRIVE = '/content/drive/MyDrive/eegmmidb-1.0.0.zip'   # zip PhysioNet déposé sur Drive
DATA_DIR  = '/content/eegmmidb-1.0.0/files'                 # après décompression
OUT_DIR   = '/content/drive/MyDrive/eeg_resultats'          # résultats + poids (sur Drive)
CACHE_DIR = '/content/cache'                                 # cache .npz local (rapide, refait après déconnexion)
REPO_URL  = 'https://github.com/kali919/EEG_Classification.git'
REPO_DIR  = '/content/EEG_Classification'

## Décompression des données (une seule fois par session)

In [ ]:
import os
if not os.path.isdir(DATA_DIR):
    # Le zip contient un dossier 'files/' à la racine -> /content/eegmmidb-1.0.0/files
    !unzip -q -n "$ZIP_DRIVE" -d /content/eegmmidb-1.0.0
n_sujets = len([d for d in os.listdir(DATA_DIR) if d.startswith('S')])
print(DATA_DIR, ':', n_sujets, 'dossiers de sujets')

## Code du projet et dépendances

In [ ]:
if os.path.isdir(REPO_DIR):
    !git -C "$REPO_DIR" pull
else:
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR
!pip install -q mne

In [ ]:
import torch
print('PyTorch', torch.__version__)
print('CUDA disponible :', torch.cuda.is_available())
assert torch.cuda.is_available(), "Pas de GPU : Exécution > Modifier le type d'exécution > GPU"
print('GPU :', torch.cuda.get_device_name(0))

## Vérification des données (105 sujets, 0 fichier manquant attendu)

In [ ]:
exclus = {88, 92, 100, 104}
runs = [1, 4, 6, 8, 10, 12, 14]
sujets = [s for s in range(1, 110) if s not in exclus]
manquants = [f"S{s:03d}R{r:02d}" for s in sujets for r in runs
             if not os.path.exists(f"{DATA_DIR}/S{s:03d}/S{s:03d}R{r:02d}.edf")]
print(len(sujets), 'sujets attendus,', len(manquants), 'fichiers manquants', manquants[:5])

## 1) Essai rapide (10 sujets, 5 époques) : vérifie que tout tourne et mesure le temps

In [ ]:
!python -W ignore train.py --model cd_cnn --classes 2 4 --quick     --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"

## 2) Entraînement complet (105 sujets, 100 époques max)

**À lancer seulement après validation de l'essai rapide.** Relancer cette cellule après une
déconnexion reprend au premier pli non terminé.

In [ ]:
!python -W ignore train.py --model cd_cnn --classes 2 4     --data-dir "$DATA_DIR" --out-dir "$OUT_DIR" --cache-dir "$CACHE_DIR"

In [ ]:
import json
for nom in ['cd_cnn_2cl', 'cd_cnn_4cl']:
    f = f'{OUT_DIR}/{nom}/resume.json'
    if os.path.exists(f):
        r = json.load(open(f, encoding='utf-8'))
        print(nom, f"global {r['global_moy']:.2%} ± {r['global_std']:.2%}")